# Phase 2: Exploratory Data Analysis (EDA)

This notebook performs the Exploratory Data Analysis (EDA) for **FraudShield** using the IEEE-CIS Fraud Detection dataset.

### Objectives:
1. **Class Distribution**: Analyze the target variable (`isFraud`) imbalance.
2. **Missing Values**: Examine columns with high missing rates and plan validation/imputation.
3. **Transaction Amounts**: Compare distributions of transaction amounts for clean vs. fraudulent events.
4. **Categorical features**: Investigate fraud rates across different card networks (`card4`) and card types (`card6`).
5. **Temporal Patterns**: Extract and visualize fraud rates by hour of the day and day of the week derived from `TransactionDT`.

In [ ]:
import sys
from pathlib import Path

# Add the project root directory to sys.path so we can import modules from src
sys.path.append(str(Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_ingestion import load_ieee_cis_dataset

sns.set_theme(style="whitegrid")
print("Libraries loaded successfully.")

## Data Loading & Ingestion Summary

In [ ]:
# Load the training split using the ingestion pipeline implemented in Phase 2
df, summary = load_ieee_cis_dataset("train")

print("Ingestion Summary:")
for k, v in summary.to_dict().items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v:,}" if isinstance(v, int) else f"  {k}: {v}")

## 1. Class Distribution Analysis

Let's measure the class imbalance in the target variable `isFraud`.

In [ ]:
fraud_counts = df['isFraud'].value_counts()
fraud_rate = df['isFraud'].mean() * 100

print(f"Non-Fraudulent Transactions (0): {fraud_counts[0]:,}")
print(f"Fraudulent Transactions (1): {fraud_counts[1]:,}")
print(f"Overall Fraud Rate: {fraud_rate:.4f}%")

# Visualize the class imbalance
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='isFraud', hue='isFraud', palette='Set2', legend=False)
plt.title("Class Distribution (isFraud)")
plt.xlabel("Is Fraud")
plt.ylabel("Count")
plt.show()

## 2. Missing Values Analysis

The IEEE-CIS dataset contains many columns with high percentages of missing values due to privacy masking and different product schemas. Let's analyze the distribution of missingness across columns.

In [ ]:
missing_pct = df.isnull().mean() * 100
missing_df = pd.DataFrame({'Missing %': missing_pct}).sort_values(by='Missing %', ascending=False)

print("Top 15 Columns with Highest Missing Values (%):")
print(missing_df.head(15))

# Distribution plot of missingness percentages across columns
plt.figure(figsize=(10, 5))
sns.histplot(missing_pct, bins=30, kde=True, color='skyblue')
plt.title("Distribution of Missing Values Percentage Across Columns")
plt.xlabel("Missing %")
plt.ylabel("Frequency (Number of Columns)")
plt.show()

## 3. Transaction Amount Analysis

Let's compare the transaction amounts (`TransactionAmt`) for fraudulent vs clean transactions.

In [ ]:
print("Transaction Amount Statistics for Clean Transactions:")
print(df[df['isFraud'] == 0]['TransactionAmt'].describe())

print("\nTransaction Amount Statistics for Fraud Transactions:")
print(df[df['isFraud'] == 1]['TransactionAmt'].describe())

# Boxplot (excluding extreme outliers to see the bulk distribution)
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='isFraud', y='TransactionAmt', hue='isFraud', palette='Set2', showfliers=False)
plt.title("Transaction Amount Distribution by Class (Excluding Outliers)")
plt.xlabel("Is Fraud")
plt.ylabel("Transaction Amt ($)")
plt.show()

# KDE plot with log scaling to handle skewness
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df[df['isFraud'] == 0], x='TransactionAmt', label='Clean', fill=True, log_scale=True, alpha=0.4, color='green')
sns.kdeplot(data=df[df['isFraud'] == 1], x='TransactionAmt', label='Fraud', fill=True, log_scale=True, alpha=0.4, color='red')
plt.title("Log-Scaled Transaction Amount Density by Class")
plt.xlabel("Transaction Amt ($) - Log Scale")
plt.ylabel("Density")
plt.legend()
plt.show()

## 4. Categorical Features Analysis

Let's examine how the fraud rate changes across different categories for card network (`card4`) and card type (`card6`).

In [ ]:
for col in ['card4', 'card6']:
    if col in df.columns:
        # Calculate fraud rate and counts per category
        stats = df.groupby(col).agg(
            total_count=('TransactionID', 'count'),
            fraud_count=('isFraud', 'sum'),
            fraud_rate=('isFraud', 'mean')
        ).reset_index()
        stats['fraud_rate'] = stats['fraud_rate'] * 100
        stats = stats.sort_values(by='total_count', ascending=False)
        
        print(f"\nFraud Stats by Category: {col}")
        print(stats.to_string(index=False))
        
        # Bar plot of fraud rates
        plt.figure(figsize=(8, 4))
        sns.barplot(data=stats, x=col, y='fraud_rate', hue=col, palette='Set2', legend=False)
        plt.title(f"Fraud Rate (%) by {col}")
        plt.ylabel("Fraud Rate (%)")
        plt.xlabel(col)
        plt.xticks(rotation=45)
        plt.show()

## 5. Temporal Patterns Analysis

`TransactionDT` in the raw dataset represents relative seconds from a reference point. Let's convert this to hour of day (0-23) and day of the week (0-6) to check if there are daily or weekly patterns in fraud rate.

In [ ]:
# Deriving temporal features
df['day_number'] = (df['TransactionDT'] // 86400).astype(int)
df['hour'] = ((df['TransactionDT'] % 86400) // 3600).astype(int)
df['day_of_week'] = (df['day_number'] % 7).astype(int)

# Group by hour
hourly_stats = df.groupby('hour')['isFraud'].mean().reset_index()
hourly_stats['fraud_rate_pct'] = hourly_stats['isFraud'] * 100

plt.figure(figsize=(12, 5))
sns.lineplot(data=hourly_stats, x='hour', y='fraud_rate_pct', marker='o', color='purple')
plt.title("Fraud Rate (%) by Hour of Day")
plt.xlabel("Hour of Day (0-23)")
plt.ylabel("Fraud Rate (%)")
plt.xticks(range(24))
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Group by day of week
daily_stats = df.groupby('day_of_week')['isFraud'].mean().reset_index()
daily_stats['fraud_rate_pct'] = daily_stats['isFraud'] * 100

plt.figure(figsize=(8, 4))
sns.barplot(data=daily_stats, x='day_of_week', y='fraud_rate_pct', hue='day_of_week', palette='Set2', legend=False)
plt.title("Fraud Rate (%) by Day of Week")
plt.xlabel("Day of Week (0 = Sunday, 6 = Saturday)")
plt.ylabel("Fraud Rate (%)")
plt.show()

## Final Summary

*(Please run the cells in this notebook to compute and display all metrics and visualizations, then record the numerical findings here.)*